In [1]:
# 1.Download the FER2013 facial expression dataset and write a Python script using pandas to load the CSV file and print the number of total samples,
# as well as the names of all unique emotion classes present.
    
import pandas as pd
import numpy as np

df = pd.read_csv('fer2013.csv')

print('Total samples:', len(df))
print('Emotion classes:', sorted(df['emotion'].unique()))


Total samples: 35887
Emotion classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


In [2]:
emotion_names = {
    0: 'Angry',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happy',
    4: 'Sad',
    5: 'Surprise',
    6: 'Neutral'
}

print(df['emotion'].map(emotion_names).unique())


['Angry' 'Fear' 'Sad' 'Neutral' 'Happy' 'Surprise' 'Disgust']


In [3]:
#2.Using numpy and pandas, calculate and display the number of samples for each emotion class in the FER2013 dataset to identify any data imbalance.<br><br><em><strong>Hint:
# </strong> Use value_counts() on the label column.</em>

counts = df['emotion'].value_counts().sort_index()
counts.index = counts.index.map(emotion_names)
print(counts)


emotion
Angry       4953
Disgust      547
Fear        5121
Happy       8989
Sad         6077
Surprise    4002
Neutral     6198
Name: count, dtype: int64


In [4]:
#3.Write a Python function that takes a single pixel string from the FER2013 dataset, converts it to a numpy array, and reshapes it into the correct image dimensions. 
# Display the shape to confirm it matches the expected size.

def convert_pixels(pixel_string):
    pixels = np.array(pixel_string.split(), dtype=np.uint8)
    return pixels.reshape(48, 48)

image = convert_pixels(df['pixels'].iloc[0])
print('Image shape:', image.shape)


Image shape: (48, 48)


In [6]:
# 4.Implement a preprocessing step that normalizes the pixel values of all images in the FER2013 dataset to the range [0, 1] and splits the data into train, validation, and test sets using an 80-10-10 split.
# <br><br><em><strong>Constraint:</strong> Use sklearn's train_test_split for both splits.</em>

from sklearn.model_selection import train_test_split

X = np.array([convert_pixels(pixels) for pixels in df["pixels"]])
X = X.astype("float32") / 255.0

y = df["emotion"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (28709, 48, 48) (28709,)
Validation: (3589, 48, 48) (3589,)
Test: (3589, 48, 48) (3589,)


In [7]:
#5.Suppose you notice that the 'disgust' class in the FER2013 dataset has very few samples. Propose one method to address this
# imbalance and write a short Python code snippet to demonstrate how you would apply it on the training set.

from sklearn.utils import resample

train_df = pd.DataFrame({'emotion': y_train})
disgust_df = train_df[train_df['emotion'] == 1]
   

In [8]:
majority_df = train_df[train_df['emotion'] != 1]

disgust_upsampled = resample(
    disgust_df,
    replace=True,
    n_samples=majority_df['emotion'].value_counts().max(),
    random_state=42
)

balanced_train = pd.concat([majority_df, disgust_upsampled])
print(balanced_train['emotion'].value_counts().sort_index())


emotion
0    3962
1    7191
2    4097
3    7191
4    4861
5    3202
6    4958
Name: count, dtype: int64
